# Training — 4 Epochs Cross-Attention Fine-Tuning


In [ ]:
!pip install --no-deps segmentation-models-pytorch==0.5.0

In [ ]:
import sys, os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import pandas as pd
import matplotlib.pyplot as plt
import multiprocessing as mp
import pickle
from timeit import default_timer as timer

# Your modified file takes priority
sys.path.insert(0, '/kaggle/input/datasets/zahouaniyacine/my-stage2-lead-model')
sys.path.append('/kaggle/input/datasets/takashisomeya/physionet-final-submission-models')
sys.path.append('/kaggle/input/datasets/hengck23/hengck23-demo-submit-physionet')

In [ ]:
DEVICE     = 'cuda'
FLOAT_TYPE = torch.float16

KAGGLE_DIR = '/kaggle/input/competitions/physionet-ecg-image-digitization'
WEIGHT_DIR = '/kaggle/input/datasets/takashisomeya/physionet-final-submission-models'
OUT_DIR    = '/kaggle/working/outputs'
SAVE_DIR   = '/kaggle/working/checkpoints'

os.makedirs(OUT_DIR,  exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

# Training config
TARGET_TOTAL_EPOCHS  = 4
EPOCHS_THIS_SESSION  = 1       # 1 epoch per Kaggle run
LR                   = 3e-5
BATCH_SIZE           = 1
SAVE_EVERY_STEPS     = 1000    # emergency checkpoint
RESUME_CKPT = '/kaggle/input/datasets/zahouaniyacine/attention-checkpoints-3/cross_attn_b6_last_full_2.pth'

WINDOW_SIZE = 240
OFFSET      = 416
xscale = 5000 / (2080 - 118)
addx   = 1
yscale = 1
IMGH, IMGW = int(1700 * yscale), int(2200 * xscale + addx)
x0, x1 = 0, 5600
y0, y1 = 0, 1696
zero_mv = [703.5, 987.5, 1271.5, 1531.5]
print('constants ok')

In [ ]:
import stage2_lead_model
print('Loading from:', stage2_lead_model.__file__)

from stage2_lead_model import Net as LeadModel
from stage2_smp_model  import Net as WholeModel
from stage2_common     import *
from stage2_model      import prob_to_series_by_max
print('imports ok')

In [ ]:
MASK_DIR = '/kaggle/input/notebooks/m1h4wk22/generate-pseudo-masks/output/masks'
RECT_DIR = '/kaggle/input/notebooks/m1h4wk22/generate-pseudo-masks/output/rectified'

valid_id = sorted([
    f.replace('.mask-coo.npz', '')
    for f in os.listdir(MASK_DIR)
    if f.endswith('.mask-coo.npz')
])
FAIL_ID = []
print('nb training samples =', len(valid_id))

In [ ]:
def read_images(path):
    image = cv2.imread(path, cv2.IMREAD_COLOR)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (IMGW, IMGH), interpolation=cv2.INTER_LINEAR)
    trim_image = image.copy()[OFFSET:y1, x0:x1]
    image = image[y0:y1, x0:x1]
    H, W, _ = image.shape
    lead_images = []
    for zmv in zero_mv:
        h0, h1 = int(zmv - WINDOW_SIZE), int(zmv + WINDOW_SIZE)
        src_h0, src_h1 = max(0, h0), min(H, h1)
        dst_h0 = src_h0 - h0
        dst_h1 = dst_h0 + (src_h1 - src_h0)
        lead_img = np.zeros((WINDOW_SIZE * 2, W, 3), np.uint8)
        lead_img[dst_h0:dst_h1] = image[src_h0:src_h1]
        lead_images.append(lead_img)
    return trim_image, np.stack(lead_images)

In [ ]:
def load_sparse_mask(path, shape=(4, 1700, 5600)):
    """Loads a sparse COO .npz mask → dense float32 (4, H, W)."""
    d = np.load(path)
    H, W = int(d['shape'][1]), int(d['shape'][2])
    mask = np.zeros((4, H, W), dtype=np.float32)
    for ch in range(4):
        if f'ch{ch}_y' in d and len(d[f'ch{ch}_y']) > 0:
            mask[ch, d[f'ch{ch}_y'], d[f'ch{ch}_x']] = d[f'ch{ch}_v']
    return mask

class Stage2AttentionDataset(Dataset):
    def __init__(self, sample_ids, fail_ids=None):
        self.ids = [s for s in sample_ids if s not in (fail_ids or [])]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        sample_id = self.ids[idx]
        _, lead_images = read_images(f'{RECT_DIR}/{sample_id}.rect.jpg')
        image_tensor = torch.from_numpy(lead_images.transpose(0, 3, 1, 2)).byte()
        mask = load_sparse_mask(f'{MASK_DIR}/{sample_id}.mask-coo.npz')
        mask_tensor = torch.from_numpy(mask).unsqueeze(1)
        return {'image': image_tensor, 'mask': mask_tensor, 'id': sample_id}

In [ ]:
# Build model
model = LeadModel(
    encoder_name='tu-timm/tf_efficientnet_b6.ns_jft_in1k',
    encoder_weights=None,
    fusion_type='cross_attn',
)

if RESUME_CKPT and os.path.exists(RESUME_CKPT):
    state = torch.load(RESUME_CKPT, map_location='cpu')
    model.load_state_dict(state, strict=False)
    print('Resumed from', RESUME_CKPT)
else:
    state = torch.load(
        f'{WEIGHT_DIR}/series_b6_shared_conv2d_lb23.10.pth',
        map_location='cpu'
    )
    model.load_state_dict(state, strict=False)
    print('Loaded from base weights')

model.to(DEVICE)
model.output_type = ['loss', 'infer']

In [ ]:
dataset = Stage2AttentionDataset(valid_id, fail_ids=FAIL_ID)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS_THIS_SESSION * len(loader))
scaler    = torch.cuda.amp.GradScaler()

print(f'Training {EPOCHS_THIS_SESSION} epoch(s), {len(loader)} steps/epoch')

In [ ]:
start_timer = timer()
model.train()

for epoch in range(EPOCHS_THIS_SESSION):
    total_loss = 0.0
    for step, batch in enumerate(loader):
        image  = batch['image'].to(DEVICE)
        mask   = batch['mask'].to(DEVICE).float()

        with torch.amp.autocast('cuda', dtype=FLOAT_TYPE):
            output = model({'image': image, 'mask': mask})
            loss   = output['loss'].mean()

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()
        total_loss += loss.item()

        if (step + 1) % SAVE_EVERY_STEPS == 0:
            ckpt_path = f'{SAVE_DIR}/cross_attn_b6_step{step+1}.pth'
            torch.save(model.state_dict(), ckpt_path)
            print(f'\n[step {step+1}] checkpoint saved → {ckpt_path}')

        elapsed = timer() - start_timer
        print(f'\r[epoch {epoch+1}] step {step+1}/{len(loader)} | loss {loss.item():.4f} | {elapsed/60:.1f} min', end='', flush=True)

    avg_loss = total_loss / len(loader)
    print(f'\n[epoch {epoch+1}] avg_loss={avg_loss:.4f}')
    last_ckpt = f'{SAVE_DIR}/cross_attn_b6_last_full_{epoch+1}.pth'
    torch.save(model.state_dict(), last_ckpt)
    print(f'Saved epoch checkpoint → {last_ckpt}')

print('Training done.')